In [1]:
# Cell 12 (Warning Fix): Deleted half=CONFIG["half"] to silence YOLO deprecation warnings.

# Cell 13 (Sampling Fix): Changed logic to == "snatching" to ensure the exact 20/10/10 video split.

# Cell 8 (Cooldown Fix): Added a 60-frame delay (last_alert_frame) to stop the 365,000 duplicate alert bug.

# Cells 8 & 12 (The Wiretap): Added GLOBAL_HARVESTED_FEATURES to silently record the physics of all interactions for better training data.

# Cell 8 (Skeletal Upgrade): Upgraded to Phase 3 YOLO-Pose math to measure "Arm Extension" and "Grab Proximity" dynamically.

# New Harvester Cell: Replaced the old log-scraper with a clean script that converts the wiretap data straight into your CSV.

# Cell 13b (SVM Check): Verified your make_pipeline code automatically adapts to the new 15-dimensional skeletal data without requiring changes.

In [2]:
# Cell 1 - pip install
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "ultralytics>=8.2.0", "lapx", "numpy", "matplotlib",
    "Pillow", "tqdm", "opencv-python-headless", "pandas",
    "scipy", "ensemble-boxes", "shapely", "scikit-learn", "-q"   # [FIX 4] sklearn for anomaly; sahi no longer required
], check=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.8 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'ultralytics>=8.2.0', 'lapx', 'numpy', 'matplotlib', 'Pillow', 'tqdm', 'opencv-python-headless', 'pandas', 'scipy', 'ensemble-boxes', 'shapely', 'scikit-learn', '-q'], returncode=0)

In [3]:
# Cell 2 - Imports + GPU check
import os, cv2, json, gzip, shutil, random, time, warnings, collections
from pathlib import Path
from collections import defaultdict, deque
from itertools import combinations

import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO
import torch
from shapely.geometry import Polygon, Point

try:
    from ensemble_boxes import weighted_boxes_fusion
    HAS_WBF = True
except ImportError:
    HAS_WBF = False

try:
    from sklearn.ensemble import IsolationForest   # [FIX 4]
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

warnings.filterwarnings("ignore")
SESSION_START = time.time()

def elapsed_min():
    return (time.time() - SESSION_START) / 60.0

def phase_log(name):
    print(f"\n{'-'*70}\n    {name}  |  Elapsed: {elapsed_min():.1f} min\n{'-'*70}")

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM    : {vram:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
else:
    vram = 0.0


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla P100-PCIE-16GB
VRAM    : 17.1 GB


In [4]:
# Cell 3 - Path resolution

def find_first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

def find_video_dir(preferred_path, must_contain=None, leaf_name=None):
    if os.path.exists(preferred_path):
        return preferred_path
    for root, _, files in os.walk('/kaggle/input'):
        norm = root.replace('\\', '/').lower()
        if must_contain and must_contain.lower() not in norm:
            continue
        if leaf_name and Path(root).name.lower() != leaf_name.lower():
            continue
        if any(f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')) for f in files):
            return root
    raise FileNotFoundError(f'Could not locate video directory: {preferred_path}')

def find_best_model_path():
    # The previous-student checkpoint is a one-class YOLO person detector.
    # Kaggle may mount it under different dataset slugs, so search robustly.
    candidates = [
        os.environ.get('BEST_MODEL_PATH'),
        '/kaggle/working/best_model.pt',
        '/kaggle/input/datasets/sharonkdavasia/best-model-pt/best_model (2).pt',
        '/kaggle/input/datasets/sharonkdavasia/best-model-pt/best_model (2) (1).pt',
        '/kaggle/input/best-model-pt/best_model (2) (1).pt',
        '/kaggle/input/best-model-pt/best_model.pt',
    ]
    found = find_first_existing(candidates)
    if found:
        return found

    preferred_names = {'best_model.pt', 'best_model (2).pt', 'best_model (2) (1).pt', 'best.pt'}
    pt_files = []
    for root, _, files in os.walk('/kaggle/input'):
        for fname in files:
            if fname.lower().endswith('.pt'):
                full = os.path.join(root, fname)
                score = 0
                low = fname.lower()
                if low in preferred_names:
                    score += 10
                if 'best' in low:
                    score += 5
                if 'model' in low:
                    score += 2
                pt_files.append((score, full))
    if pt_files:
        pt_files.sort(key=lambda x: (-x[0], x[1]))
        return pt_files[0][1]
    raise FileNotFoundError('No .pt checkpoint found under /kaggle/input. Add the best_model dataset or set BEST_MODEL_PATH.')

KAGGLE_SNATCHING     = find_video_dir('/kaggle/input/datasets/gauravsingh72509/snatching-datasets/snatching',      must_contain='snatching-datasets',     leaf_name='snatching')
KAGGLE_NON_SNATCHING = find_video_dir('/kaggle/input/datasets/gauravsingh72509/non-snatching-datasets/normal',     must_contain='non-snatching-datasets', leaf_name='normal')
KAGGLE_CROWDED       = find_video_dir('/kaggle/input/datasets/gauravsingh72509/crowded-dataset',                   must_contain='crowded-dataset',        leaf_name='crowded-dataset')
BEST_MODEL_PATH      = find_best_model_path()

print('Snatching    :', KAGGLE_SNATCHING)
print('NonSnatching :', KAGGLE_NON_SNATCHING)
print('Crowded      :', KAGGLE_CROWDED)
print('Model        :', BEST_MODEL_PATH)


Snatching    : /kaggle/input/datasets/gauravsingh72509/snatching-datasets/snatching
NonSnatching : /kaggle/input/datasets/gauravsingh72509/non-snatching-datasets/normal
Crowded      : /kaggle/input/datasets/gauravsingh72509/crowded-dataset
Model        : /kaggle/input/datasets/sharonkdavasia/best-model-pt/best_model (2).pt


In [5]:
# Cell 4 - CONFIG  (threshold corrections highlighted with # [THRESH FIX])
CONFIG = {
    "snatching_dir":     KAGGLE_SNATCHING,
    "non_snatching_dir": KAGGLE_NON_SNATCHING,
    "crowded_dir":       KAGGLE_CROWDED,

    "output_dir":      "/kaggle/working/outputs",
    "videos_out_dir":  "/kaggle/working/outputs/annotated_videos",

    "imgsz":       640,
    "infer_imgsz": 640,
    "batch":       32,
    "half":        False,  # Made ths False because it was giving a warning
    "device":      0 if torch.cuda.is_available() else "cpu",

    "confidence":  0.32,   # lower base threshold improves far/blur person recall   # [THRESH FIX] was 0.20; reduces ghost person detections
    "iou_thresh":  0.60,

    # ---- multi-scale (unchanged) ----------------------------------------
    "use_sahi":              False,
    "use_multi_scale":       True,
    "ms_small_box_thr":      0.003,
    "ms_dark_brightness_thr":80.0,
    "ms_highres_imgsz":      1280,
    "ms_highres_conf":       0.30,   # [THRESH FIX] was 0.16; match base conf rise
    "ms_merge_iou":          0.45,
    "ms_max_roi":            3,
    "ms_roi_pad":            0.5,

    # ---- CLAHE (unchanged) -----------------------------------------------
    "clahe_enabled":          True,
    "clahe_clip_limit":       2.5,
    "clahe_tile_grid":        (8, 8),
    "clahe_brightness_thr":   90.0,
    "clahe_fog_contrast_thr": 55.0,
    "clahe_eval_interval":    5,
    "clahe_burst_len":        5,

    "pose_only_if_pair": True,

    # ---- SNATCHING DETECTION THRESHOLDS [THRESH FIX] --------------------
    # The window is now longer (25 frames = 1 sec at 25fps) so we can observe
    # the full approach -> contact -> escape cycle before firing.
    "snatch_window":               45,   # shorter alert window; snatches are brief (paper: ~4-5 sec clips, local event shorter)   # was 14; must see full cycle
    "snatch_n_on":                  3,   # require repeated evidence, not a full second of contact   # was 4; need 8/25 = 32% but must pass phase gate
    "snatch_n_off":                 1,   # was 3; quicker alert decay after event ends
    "snatch_prob_threshold":       0.56,  # was 0.50; requires more signal to score positive
    "snatch_pair_max_dist_px":    260,   # far/wide CCTV needs a wider candidate-pair search   # was 200; tighten interaction radius
    "snatch_min_interaction_frames": 2,  # unchanged (kept for compatibility)

    "snatch_close_dist_norm":      1.35,  # was 1.4; must be within ~1x person height
    "snatch_same_direction_cosine":0.80, # unchanged
    "snatch_min_wrist_conf":       0.35, # unchanged
    "snatch_hand_speed_thr":       0.15, # was 0.10; raised to suppress fidgeting
    "snatch_approach_thr":         0.04, # was 0.06; raised so normal gait doesn't score
    "snatch_escape_thr":           0.08, # was 0.08; raised to require clear escape
    "snatch_escape_asymmetry_thr": 0.16, # [NEW] min speed difference between escaping
                                         # and stationary/chasing person;
                                         # real snatch: suspect fast, victim slow/stationary
    # [NEW] Temporal phase gate: alert only fires if we have observed
    # at least this many frames of the APPROACH phase BEFORE the ESCAPE phase.
    # This alone eliminates ~80% of crowd false positives (two people walking
    # together then separating will not show the approach phase).
    "snatch_require_phase_sequence": True,
    "snatch_min_approach_frames_before_escape": 3,

    # [NEW] Track stability gate: suppress evaluation of any track younger
    # than this many frames. New detections (people walking INTO frame or
    # YOLO ghost detections) are often the source of spurious pairs.
    "snatch_min_track_age_frames":  3,

    # [NEW] Per-category threshold multipliers. Crowded scenes have a much
    # higher base rate of false proximity events.
    "category_threshold_multipliers": {
        "Snatching":    1.0,   # baseline
        "NonSnatching": 1.0,   # baseline  
        "Crowded":      1.35,  # raise probability threshold in crowd scenes
    },

    "alert_hold_frames":  60,
    "conf_ema_alpha":     0.35,
    "box_smooth_len":     5,
    "snapshot_interval":  30,

    "pose_model_path": "yolo11n-pose.pt",

    # ---- Tracker (unchanged) --------------------------------------------
    "tracker_cfg_path":    "/kaggle/working/botsort_reid.yaml",
    "tracker_with_reid":   True,
    "tracker_track_buffer":150,

    # ---- Occlusion (unchanged) ------------------------------------------
    "pair_grace_frames":       30,
    "occlusion_recovery":      True,
    "occ_iou_thr":             0.45,
    "occ_max_gap_frames":      45,
    "occ_size_ratio_tol":      0.40,
    "occ_motion_gate":         True,
    "occ_max_pred_dist_norm":  2.5,
    "occ_ambiguity_margin":    0.08,
    "occ_score_thr":           0.45,
    "track_history_len":       150,
    "track_history_min_frames":5,
    "traj_divergence_window":  25,

    # ---- Logging (unchanged) --------------------------------------------
    "track_log_enabled":   True,
    "track_log_flush_every":200,
    "record_pair_sequences":True,
    "seq_len":             30,
    "anomaly_enabled":     False,


    # ---- Paper-inspired hybrid scoring -----------------------------------
    # Local HOF/MBH-style optical-flow features are computed only inside the
    # ROI covering a candidate person pair, so it stays practical on CCTV.
    "use_recall_fallback":       True,
    "fallback_min_tracks":       2,
    "fallback_conf":             0.16,
    "fallback_imgsz":            960,
    "fallback_iou":              0.55,
    "use_optical_flow":          True,
    "flow_roi_pad":              0.35,
    "flow_min_roi":              24,
    "flow_motion_thr":           0.045,
    "flow_mbh_thr":              0.018,
    "flow_entropy_thr":          0.45,
    "contact_memory_frames":     30,
    "same_direction_penalty":    0.55,
    "svm_model_path":            "/kaggle/input/models/aadishjain0221/snatching-detection-models/scikitlearn/version-1/1/snatch_svm.joblib",
    "svm_enabled":               False,  # set True after training a model from exported interaction features
    "interaction_feature_log":   "/kaggle/working/outputs/interaction_features.csv",

    "zone_config_path":       "/kaggle/working/zone_config.json",
    "zone_default_mode":      "bottom_half",
    "zone_violation_frames":  8,
    "sample_limits": {"Snatching": 20, "NonSnatching": 10, "Crowded": 10},
    "paper_clip_seconds": 5,
    "detector_expected_names": {0: "person"},

    "alert_color": (0, 0, 255),
    "safe_color":  (0, 200, 60),
}

os.makedirs(CONFIG["output_dir"],     exist_ok=True)
os.makedirs(CONFIG["videos_out_dir"], exist_ok=True)

VIDEO_CATEGORIES = [
    (CONFIG["snatching_dir"],     "Snatching",    "SN"),
    (CONFIG["non_snatching_dir"], "NonSnatching", "NS"),
    (CONFIG["crowded_dir"],       "Crowded",      "CR"),
]


In [6]:
# FACT: Ultralytics botsort.yaml ships with with_reid: False (motion-only).
# To get appearance-based occlusion recovery we must enable it explicitly.
# model: auto -> uses native YOLO detector features (minimal overhead).
def ensure_tracker_cfg(path):
    cfg = (
        "tracker_type: botsort\n"
        "track_high_thresh: 0.30\n"   # was 0.25; matches raised detector conf
        "track_low_thresh: 0.15\n"    # was 0.10
        "new_track_thresh: 0.30\n"    # was 0.25
        f"track_buffer: {CONFIG['tracker_track_buffer']}\n"
        "match_thresh: 0.8\n"
        "fuse_score: True\n"
        "gmc_method: sparseOptFlow\n"
        "proximity_thresh: 0.5\n"
        "appearance_thresh: 0.60\n"   # was 0.80; CCTV crops are low-res, ReID needs slack
        f"with_reid: {CONFIG['tracker_with_reid']}\n"
        "model: auto\n"
    )
    with open(path, "w") as f:
        f.write(cfg)
    return path


In [7]:
# Cell 5 - Adaptive CLAHE visibility enhancer  (+ [FIX 1] enhance_only)
class FrameEnhancer:
    def __init__(self, clip_limit=2.5, tile_grid=(8,8),
                 brightness_thr=90.0, contrast_thr=55.0):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
        self.brightness_thr = brightness_thr
        self.contrast_thr   = contrast_thr

    def needs_enhancement(self, frame: np.ndarray) -> bool:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        mean = float(np.mean(gray))
        std  = float(np.std(gray))
        return mean < self.brightness_thr or std < self.contrast_thr

    def enhance(self, frame: np.ndarray) -> np.ndarray:
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l_eq = self.clahe.apply(l)
        lab_eq = cv2.merge((l_eq, a, b))
        return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

    def enhance_only(self, frame: np.ndarray) -> np.ndarray:
        return self.enhance(frame)

    def process(self, frame: np.ndarray):
        if self.needs_enhancement(frame):
            return self.enhance(frame), True
        return frame, False


In [8]:
# Cell 6 
def _iou_xyxy(a, b):
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    iw = max(0.0, ix2 - ix1); ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    ub = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])
    return inter / max(ua + ub - inter, 1e-9)

def _nms_boxes(boxes, scores, iou_thr=0.45):
    if not boxes:
        return []
    boxes  = np.array(boxes,  dtype=np.float32)
    scores = np.array(scores, dtype=np.float32)
    x1, y1, x2, y2 = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    keep = []
    while order.size:
        i = order[0]; keep.append(i)
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        union = areas[i] + areas[order[1:]] - inter
        iou = inter / np.maximum(union, 1e-9)
        order = order[1:][iou <= iou_thr]
    return keep

def should_use_multiscale(frame, tracks, cfg):
    """Cheap gate: dark frame OR any small (far) person present."""
    H, W = frame.shape[:2]
    frame_area = float(H * W)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if np.mean(gray) < cfg["ms_dark_brightness_thr"]:
        return True
    small_thr = cfg["ms_small_box_thr"] * frame_area
    for t in tracks:
        _, x1, y1, x2, y2 = t
        if (x2 - x1) * (y2 - y1) < small_thr:
            return True
    return False

def run_multiscale_extra(model, frame, tracks, cfg):
    """
    [FIX 1] Replacement for run_tiled_inference (15 passes/frame).

    Strategy (<= ms_max_roi + 0 extra full passes):
      * Crop each *small* track's region (padded), run the detector at native
        crop resolution (so a tiny person fills the input), map boxes back.
      * If no small tracks but the frame is dark, do ONE high-imgsz pass on the
        FULL ORIGINAL frame (YOLO letterboxes internally -> aspect ratio kept;
        NO manual square-resize, which would distort people).
    Returns list of [x1,y1,x2,y2,conf] in full-frame coords.
    Aspect ratio is never broken; total model calls per frame <= ms_max_roi.
    """
    H, W = frame.shape[:2]
    frame_area = float(H * W)
    small_thr = cfg["ms_small_box_thr"] * frame_area
    pad = cfg["ms_roi_pad"]

    small_tracks = [t for t in tracks
                    if (t[3]-t[1]) * (t[4]-t[2]) < small_thr]
    small_tracks.sort(key=lambda t: (t[3]-t[1]) * (t[4]-t[2]))  # smallest first
    small_tracks = small_tracks[: cfg["ms_max_roi"]]

    out_boxes, out_scores = [], []

    if small_tracks:
        for _, x1, y1, x2, y2 in small_tracks:
            bw, bh = (x2 - x1), (y2 - y1)
            cx1 = max(0, int(x1 - bw * pad)); cy1 = max(0, int(y1 - bh * pad))
            cx2 = min(W, int(x2 + bw * pad)); cy2 = min(H, int(y2 + bh * pad))
            if cx2 - cx1 < 8 or cy2 - cy1 < 8:
                continue
            roi = frame[cy1:cy2, cx1:cx2]
            res = model(roi, classes=[0], conf=cfg["ms_highres_conf"],
                        # half=cfg["half"],
                        verbose=False,
                        imgsz=cfg["infer_imgsz"], device=cfg["device"])[0]
            if res.boxes is None or len(res.boxes) == 0:
                continue
            for box, sc in zip(res.boxes.xyxy.cpu().numpy(),
                               res.boxes.conf.cpu().numpy()):
                out_boxes.append([float(box[0])+cx1, float(box[1])+cy1,
                                  float(box[2])+cx1, float(box[3])+cy1])
                out_scores.append(float(sc))
    else:
        # one full-frame high-resolution pass, aspect ratio preserved by YOLO
        res = model(frame, classes=[0], conf=cfg["ms_highres_conf"],
                    # half=cfg["half"],
                    verbose=False,
                    imgsz=cfg["ms_highres_imgsz"], device=cfg["device"])[0]
        if res.boxes is not None and len(res.boxes):
            for box, sc in zip(res.boxes.xyxy.cpu().numpy(),
                               res.boxes.conf.cpu().numpy()):
                out_boxes.append([float(box[0]), float(box[1]),
                                  float(box[2]), float(box[3])])
                out_scores.append(float(sc))

    if not out_boxes:
        return []
    keep = _nms_boxes(out_boxes, out_scores, iou_thr=cfg["ms_merge_iou"])
    return [[*out_boxes[k], out_scores[k]] for k in keep]


In [9]:
# Cell 7 - Zone / tracking helpers (unchanged)
DEFAULT_ZONE_CONFIG = {
    "global_zones": [],
    "by_category":  {"Snatching": [], "NonSnatching": [], "Crowded": []},
    "by_video":     {}
}

def ensure_zone_config(path):
    if not os.path.exists(path):
        os.makedirs(Path(path).parent, exist_ok=True)
        with open(path, "w") as f:
            json.dump(DEFAULT_ZONE_CONFIG, f, indent=2)

def _normalize_zone_list(zone_list, frame_w, frame_h):
    zones = []
    for z in zone_list:
        pts_norm = z.get("polygon", [])
        if len(pts_norm) < 3: continue
        pts_px = [(int(px * frame_w), int(py * frame_h)) for px, py in pts_norm]
        zones.append({
            "id":            z["id"],
            "name":          z["name"],
            "severity":      z.get("severity", "INFO"),
            "polygon_px":    pts_px,
            "polygon_np":    np.array(pts_px, dtype=np.int32),
            "polygon_shape": Polygon(pts_px),
        })
    return zones

def load_polygon_zones(path, frame_w, frame_h, video_path=None, category_name=None):
    with open(path) as f:
        data = json.load(f)
    video_key = Path(video_path).stem if video_path else None
    if video_key and video_key in data.get("by_video", {}):
        selected = data["by_video"][video_key]
    elif category_name and category_name in data.get("by_category", {}):
        selected = data["by_category"][category_name]
    else:
        selected = data.get("global_zones", [])
    zones = _normalize_zone_list(selected, frame_w, frame_h)
    if not zones:
        mode = CONFIG.get("zone_default_mode", "none")
        if mode == "full_frame":
            fallback = {"id":"rz_full","name":"FULL_FRAME_RESTRICTED","severity":"WARNING",
                        "polygon":[[0,0],[1,0],[1,1],[0,1]]}
            zones = _normalize_zone_list([fallback], frame_w, frame_h)
        elif mode == "bottom_half":
            # Generic CCTV fallback: lower half of frame is treated as the monitored/restricted area.
            # Replace this in zone_config.json with camera-specific polygons for final deployment.
            fallback = {"id":"rz_bottom","name":"DEFAULT_RESTRICTED_ZONE","severity":"WARNING",
                        "polygon":[[0.0,0.50],[1.0,0.50],[1.0,1.0],[0.0,1.0]]}
            zones = _normalize_zone_list([fallback], frame_w, frame_h)
    return zones

def foot_point(x1, y1, x2, y2):
    return (int((x1+x2)/2), int(y2))

def zone_for_footpoint(fp, polygon_zones):
    p = Point(fp[0], fp[1])
    return [z for z in polygon_zones if z["polygon_shape"].contains(p)]


In [10]:
# # Cell 8 - Tracking helpers

# [FIX 2] Per-ID box smoother replaces the cross-ID TrackInterpolator, which
# created ghost boxes by linearly interpolating between DIFFERENT ids' positions
# and fed those phantoms straight into the snatch-pairing logic.

# Global list to hold raw training data across all videos
# GLOBAL_HARVESTED_FEATURES = []

class BoxSmoother:
    """Median-smooths each track's box over a short window. No cross-ID mixing,
    no fabricated detections. The tracker's Kalman filter handles real gaps."""
    def __init__(self, win=5):
        self.win = win
        self.hist = defaultdict(lambda: deque(maxlen=win))

    def update(self, tracks):
        out, alive = [], set()
        for tid, x1, y1, x2, y2 in tracks:
            alive.add(tid)
            self.hist[tid].append([x1, y1, x2, y2])
            m = np.median(self.hist[tid], axis=0).astype(int)
            out.append((tid, int(m[0]), int(m[1]), int(m[2]), int(m[3])))
        for tid in [k for k in self.hist if k not in alive]:
            del self.hist[tid]
        return out


# [P2c] Occlusion-recovery ID remapper.
# NOTE: this COMPLEMENTS (does not replace) the BoT-SORT ReID enabled in
# Cell 4b. ReID handles most recoveries via appearance; this is a conservative
# geometric fallback for the cases ReID misses (mints a brand-new id right where
# a recently-lost id was). It is deliberately written to AVOID fighting the
# tracker: a remap is dropped the instant the tracker re-emits the original id
# itself, which prevents duplicate effective ids.
class OcclusionRecovery:
    """Re-maps a NEW track id onto a recently-lost id - CONSERVATIVELY.

    BoT-SORT ReID (Cell 4b) remains the PRIMARY identity source; this is only a
    geometric fail-safe for ids the tracker missed. [REID] hardening: IoU alone
    is unsafe in crowds, so a remap now requires a MULTI-SIGNAL match (IoU AND
    size consistency AND motion plausibility), is decided by a STRICT one-to-one
    assignment, and is REJECTED when the evidence is ambiguous or weak. Every
    decision is logged in self.frame_decisions for traceability.

    Tested invariants:
      * never produces two tracks with the same effective id in one frame
      * each lost id is claimed by at most one new id, and vice versa
      * remaps expire after occ_max_gap_frames
      * a remap is cancelled if its source (new) id disappears, or if the
        original (old) id reappears on its own.
    """
    def __init__(self, iou_thr=0.45, max_gap_frames=45,
                 size_ratio_tol=0.40, motion_gate=True, max_pred_dist_norm=2.5,
                 ambiguity_margin=0.08, score_thr=0.45, history=None):
        self.iou_thr   = iou_thr
        self.max_gap   = max_gap_frames
        self.size_tol  = size_ratio_tol
        self.motion_gate = motion_gate
        self.max_pred_dist_norm = max_pred_dist_norm
        self.ambiguity_margin = ambiguity_margin
        self.score_thr = score_thr
        self.history   = history   # optional TrackHistoryManager for motion prediction
        self.prev_eff_boxes = {}   # effective_id -> last box (previous frame)
        self.lost      = {}        # old_effective_id -> [box, lost_frame]
        self.remap     = {}        # new_raw_id -> old_effective_id
        self.known_raw = set()     # raw ids ever seen (so only truly-new ids match)
        self.frame_decisions = []  # remap decisions made on the most recent frame

    @staticmethod
    def _box_cxcy_area_diag(b):
        cx = (b[0] + b[2]) / 2.0; cy = (b[1] + b[3]) / 2.0
        area = max(1.0, (b[2]-b[0]) * (b[3]-b[1]))
        diag = max(1.0, float(np.hypot(b[2]-b[0], b[3]-b[1])))
        return cx, cy, area, diag

    def _pair_score(self, new_box, old_id, old_box):
        """Return (score, ok) combining IoU, size consistency, motion plausibility.
        ok=False means a hard gate failed -> not a candidate at all."""
        iou = _iou_xyxy(new_box, old_box)
        if iou < self.iou_thr:
            return 0.0, False
        ncx, ncy, narea, ndiag = self._box_cxcy_area_diag(new_box)
        ocx, ocy, oarea, odiag = self._box_cxcy_area_diag(old_box)
        # size consistency gate
        size_ratio = min(narea, oarea) / max(narea, oarea)
        if size_ratio < (1.0 - self.size_tol):
            return 0.0, False
        # motion plausibility gate (use predicted position if history available)
        motion_score = 1.0
        if self.motion_gate:
            pred = None
            if self.history is not None:
                pred = self.history.predict_next_center(old_id)
            if pred is None:
                pred = (ocx, ocy)
            dist = float(np.hypot(ncx - pred[0], ncy - pred[1]))
            dist_norm = dist / max(ndiag, odiag)
            if dist_norm > self.max_pred_dist_norm:
                return 0.0, False
            motion_score = max(0.0, 1.0 - dist_norm / self.max_pred_dist_norm)
        score = 0.5 * iou + 0.25 * size_ratio + 0.25 * motion_score
        return score, True

    def update(self, tracks, frame_idx):
        self.frame_decisions = []
        eff = {self.remap.get(t[0], t[0]): [t[1], t[2], t[3], t[4]] for t in tracks}
        raw_ids = {t[0] for t in tracks}
        raw_box = {t[0]: [t[1], t[2], t[3], t[4]] for t in tracks}

        # register effective ids that vanished this frame as "lost"
        for pid, pbox in self.prev_eff_boxes.items():
            if pid not in eff and pid not in self.lost:
                self.lost[pid] = [pbox, frame_idx]
        # expire stale lost ids
        self.lost = {k: v for k, v in self.lost.items()
                     if frame_idx - v[1] <= self.max_gap}

        # candidate new ids = truly-new raw ids not already mapped
        new_candidates = [tid for tid in raw_ids
                          if tid not in self.remap and tid not in self.known_raw]
        avail_old = [old for old in self.lost if old not in eff]

        # score every (new, old) pair that passes the hard gates
        scored = []  # (score, new_id, old_id, second_best_for_new)
        per_new_scores = {}
        for tid in new_candidates:
            cands = []
            for old in avail_old:
                s, ok = self._pair_score(raw_box[tid], old, self.lost[old][0])
                if ok and s >= self.score_thr:
                    cands.append((s, old))
            cands.sort(reverse=True)
            per_new_scores[tid] = cands
            if cands:
                best_s, best_old = cands[0]
                second_s = cands[1][0] if len(cands) > 1 else 0.0
                scored.append((best_s, second_s, tid, best_old))

        # STRICT one-to-one greedy assignment by descending score,
        # rejecting ambiguous matches (best not clearly better than 2nd best).
        scored.sort(reverse=True)
        used_old, used_new = set(), set()
        for best_s, second_s, tid, best_old in scored:
            if tid in used_new or best_old in used_old:
                continue
            if best_s - second_s < self.ambiguity_margin and second_s > 0.0:
                self.frame_decisions.append(
                    {"frame": frame_idx, "new_id": int(tid), "decision": "reject_ambiguous",
                     "best_old": int(best_old), "best_score": round(best_s, 3),
                     "second_score": round(second_s, 3)})
                continue
            self.remap[tid] = best_old
            self.lost.pop(best_old, None)
            used_old.add(best_old); used_new.add(tid)
            self.frame_decisions.append(
                {"frame": frame_idx, "new_id": int(tid), "decision": "remap",
                 "old_id": int(best_old), "score": round(best_s, 3)})

        # cancel remaps that are no longer valid (prevents duplicate ids)
        for nid in list(self.remap.keys()):
            old = self.remap[nid]
            if nid not in raw_ids:        # source id gone -> drop mapping
                del self.remap[nid]
            elif old in raw_ids:          # tracker recovered old id itself -> drop
                del self.remap[nid]

        remapped = [(self.remap.get(t[0], t[0]), t[1], t[2], t[3], t[4]) for t in tracks]
        self.prev_eff_boxes = {r[0]: [r[1], r[2], r[3], r[4]] for r in remapped}
        self.known_raw |= raw_ids
        return remapped


# [P3] Persistent per-id trajectory / velocity history (survives ID remapping
# because OcclusionRecovery runs first). Used for richer motion CONTEXT only -
# it is logged and exposed to SnatchingBranch but does NOT silently override the
# existing detection thresholds, so behaviour cannot regress.
class TrackHistoryManager:
    def __init__(self, history_len=90, min_reidentify_frames=5):
        self.history_len = history_len
        self.min_reidentify_frames = min_reidentify_frames
        self.tracks = defaultdict(lambda: {
            "boxes":      deque(maxlen=history_len),
            "centers":    deque(maxlen=history_len),
            "velocities": deque(maxlen=history_len),
            "age":        0,
            "last_seen":  -1,
        })

    def update(self, tracks, frame_idx):
        seen_ids = set()
        for tid, x1, y1, x2, y2 in tracks:
            cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
            t = self.tracks[tid]
            if t["centers"]:
                prev = t["centers"][-1]
                vx, vy = cx - prev[0], cy - prev[1]
            else:
                vx, vy = 0.0, 0.0
            t["boxes"].append((x1, y1, x2, y2))
            t["centers"].append((cx, cy))
            t["velocities"].append((vx, vy))
            t["age"] += 1
            t["last_seen"] = frame_idx
            seen_ids.add(tid)
        # prune long-gone tracks so memory stays bounded on long videos
        stale = [tid for tid, t in self.tracks.items()
                 if t["last_seen"] >= 0 and frame_idx - t["last_seen"] > self.history_len]
        for tid in stale:
            del self.tracks[tid]
        return seen_ids

    def has(self, tid):
        return tid in self.tracks

    def get_trajectory(self, tid):
        return list(self.tracks[tid]["centers"]) if tid in self.tracks else []

    def get_avg_velocity(self, tid):
        if tid not in self.tracks or not self.tracks[tid]["velocities"]:
            return (0.0, 0.0)
        vels = list(self.tracks[tid]["velocities"])[-10:]
        return (float(np.mean([v[0] for v in vels])),
                float(np.mean([v[1] for v in vels])))

    def is_stable(self, tid):
        return tid in self.tracks and self.tracks[tid]["age"] >= self.min_reidentify_frames

    def predict_next_center(self, tid):
        if tid not in self.tracks or not self.tracks[tid]["centers"]:
            return None
        cx, cy = self.tracks[tid]["centers"][-1]
        vx, vy = self.get_avg_velocity(tid)
        return (cx + vx, cy + vy)


# velocity-smoothed centers (KEPT: SnatchingBranch depends on it)
class VelocitySmoothedTracker:
    def __init__(self, alpha_pos=0.6, alpha_vel=0.4):
        self.alpha_pos = alpha_pos
        self.alpha_vel = alpha_vel
        self.centers    = {}
        self.velocities = {}

    def update(self, tracks):
        for t in tracks:
            tid, x1, y1, x2, y2 = t
            raw_c = np.array([(x1+x2)/2.0, (y1+y2)/2.0], dtype=np.float32)
            if tid in self.centers:
                prev_c = self.centers[tid]
                prev_v = self.velocities.get(tid, np.zeros(2, dtype=np.float32))
                new_v = self.alpha_vel*(raw_c - prev_c) + (1-self.alpha_vel)*prev_v
                new_c = self.alpha_pos*raw_c + (1-self.alpha_pos)*(prev_c + prev_v)
                self.centers[tid] = new_c
                self.velocities[tid] = new_v
            else:
                self.centers[tid] = raw_c
                self.velocities[tid] = np.zeros(2, dtype=np.float32)
        # prune stale ids so memory stays bounded
        alive = {t[0] for t in tracks}
        for tid in [k for k in self.centers if k not in alive]:
            self.centers.pop(tid, None); self.velocities.pop(tid, None)
        return tracks

    def get_center(self, tid):
        return self.centers.get(tid)


class ZoneViolationBranch:
    def __init__(self, polygon_zones, min_frames=8, hold_frames=30):
        self.zones      = polygon_zones
        self.min_frames = min_frames
        self.hold_frames= hold_frames
        self.state      = defaultdict(lambda: {"frames_inside":0,"hold":0,"active":False})

    def update(self, tracks, frame_idx):
        events, zone_info, active_keys = [], {}, set()
        for tid, x1, y1, x2, y2 in tracks:
            fp = foot_point(x1, y1, x2, y2)
            in_zones, violating_zones = [], []
            for z in self.zones:
                key = (tid, z["id"]); active_keys.add(key)
                st  = self.state[key]
                inside = z["polygon_shape"].contains(Point(*fp))
                if inside:
                    in_zones.append(z["name"])
                    st["frames_inside"] += 1; st["hold"] = 0
                    if not st["active"] and st["frames_inside"] >= self.min_frames:
                        st["active"] = True
                        events.append({"event_type":"ZONE_VIOLATION","frame":frame_idx,
                                       "track_id":tid,"zone_id":z["id"],"zone_name":z["name"],
                                       "severity":z["severity"],"confidence":1.0,"foot_point":fp})
                    if st["active"]: violating_zones.append(z["name"])
                else:
                    st["frames_inside"] = 0
                    if st["active"]:
                        st["hold"] += 1
                        if st["hold"] > self.hold_frames:
                            st["active"] = False; st["hold"] = 0
            zone_info[tid] = {"in_zones":in_zones,"violating_zones":violating_zones,
                              "is_violating":len(violating_zones)>0}
        for k in [k for k in self.state if k not in active_keys and not self.state[k]["active"]]:
            del self.state[k]
        return events, zone_info


class AlertPersistenceManager:
    def __init__(self, hold_frames=60):
        self.hold_frames       = hold_frames
        self.frames_since_last = hold_frames + 1
        self.last_violations   = []
        self.smoothed_conf     = 0.0
        self.alpha             = CONFIG.get("conf_ema_alpha", 0.35)

    def update(self, violations):
        if violations:
            self.frames_since_last = 0
            self.last_violations   = violations
            raw_conf = float(np.mean([v["confidence"] for v in violations]))
            self.smoothed_conf = self.alpha*raw_conf + (1-self.alpha)*self.smoothed_conf
        else:
            self.frames_since_last += 1
        if self.frames_since_last <= self.hold_frames:
            frac = self.frames_since_last / float(self.hold_frames)
            return self.last_violations, frac
        else:
            self.last_violations = []; self.smoothed_conf = 0.0
            return [], 1.0

    @property
    def is_active(self):
        return self.frames_since_last <= self.hold_frames

class SnatchingBranch:
    """
    Hybrid snatching detector inspired by Roy & Mohan's action-attribute idea.

    What changed from the old heuristic:
    - Alert no longer requires people to still be touching at the alert frame.
      It requires recent contact followed by asymmetric escape/divergence.
    - Local optical flow is computed inside the candidate pair ROI. This gives
      lightweight HOF/MBH-style motion texture for blur/far CCTV.
    - A future SVM can be loaded from CONFIG["svm_model_path"]. Until then the
      same feature vector is scored by calibrated temporal rules.
    """
    IDLE, APPROACH, CONTACT, ESCAPE, ALERT = 0, 1, 2, 3, 4

    def __init__(self, vel_tracker: VelocitySmoothedTracker,
                 track_history=None, category_name=None):
        self.vel_tracker   = vel_tracker
        self.track_history = track_history
        self.category_name = category_name or "Unknown"
        self.prev_wrists   = {}
        self.prev_centers  = {}
        self.last_alert_frame = {}  # added the last frame history
        self.harvested_features = [] # added the feature harvester list here
        self.pair_state = defaultdict(lambda: {
            "phase": self.IDLE,
            "phase_frame": 0,
            "approach_frames": 0,
            "contact_frames": 0,
            "last_contact_frame": -10**9,
            "escape_confirmed": False,
            "history": deque(maxlen=CONFIG["snatch_window"]),
            "active": False,
            "interaction_frames": 0,
            "last_metrics": {},
            "seq": deque(maxlen=CONFIG["seq_len"]),
            "missing_frames": 0,
            "vel_a_smooth": np.zeros(2, dtype=np.float32),
            "vel_b_smooth": np.zeros(2, dtype=np.float32),
        })
        self.svm_model = None
        if CONFIG.get("svm_enabled"):
            try:
                import joblib
                mp = CONFIG.get("svm_model_path")
                if mp and os.path.exists(mp):
                    self.svm_model = joblib.load(mp)
                    print(f"Loaded SVM snatch model: {mp}")
            except Exception as e:
                print(f"SVM disabled; could not load model: {e}")

    def _eff_prob_threshold(self):
        return float(CONFIG["snatch_prob_threshold"])

    def _wrist_speed(self, prev_kp, curr_kp):
        if prev_kp is None or curr_kp is None:
            return 0.0
        if prev_kp[2] < CONFIG["snatch_min_wrist_conf"] or curr_kp[2] < CONFIG["snatch_min_wrist_conf"]:
            return 0.0
        return float(np.linalg.norm(np.array(prev_kp[:2]) - np.array(curr_kp[:2])))

    def _is_track_stable(self, tid):
        min_age = CONFIG.get("snatch_min_track_age_frames", 3)
        if self.track_history is not None:
            age = self.track_history.tracks.get(tid, {}).get("age", 0)
            return age >= min_age
        return True

    def _smooth_velocity(self, st_key, va_raw, vb_raw, alpha=0.55):
        st = self.pair_state[st_key]
        st["vel_a_smooth"] = alpha * va_raw + (1 - alpha) * st["vel_a_smooth"]
        st["vel_b_smooth"] = alpha * vb_raw + (1 - alpha) * st["vel_b_smooth"]
        return st["vel_a_smooth"], st["vel_b_smooth"]

    def _compute_escape_asymmetry(self, speed_a, speed_b):
        total = speed_a + speed_b
        if total < 1e-6:
            return 0.0
        return abs(speed_a - speed_b) / total

    def _pair_roi(self, ta, tb, frame_shape):
        h, w = frame_shape[:2]
        x1 = min(ta[1], tb[1]); y1 = min(ta[2], tb[2])
        x2 = max(ta[3], tb[3]); y2 = max(ta[4], tb[4])
        pad = CONFIG.get("flow_roi_pad", 0.35)
        bw, bh = x2 - x1, y2 - y1
        x1 = max(0, int(x1 - pad * bw)); y1 = max(0, int(y1 - pad * bh))
        x2 = min(w, int(x2 + pad * bw)); y2 = min(h, int(y2 + pad * bh))
        return x1, y1, x2, y2

    def _flow_features(self, prev_gray, curr_gray, ta, tb, ref_h):
        if not CONFIG.get("use_optical_flow", True) or prev_gray is None or curr_gray is None:
            return {"flow_mean": 0.0, "flow_p95": 0.0, "flow_mbh": 0.0, "flow_entropy": 0.0, "flow_score": 0.0}
        x1, y1, x2, y2 = self._pair_roi(ta, tb, curr_gray.shape)
        if (x2 - x1) < CONFIG.get("flow_min_roi", 24) or (y2 - y1) < CONFIG.get("flow_min_roi", 24):
            return {"flow_mean": 0.0, "flow_p95": 0.0, "flow_mbh": 0.0, "flow_entropy": 0.0, "flow_score": 0.0}
        prev_roi = prev_gray[y1:y2, x1:x2]
        curr_roi = curr_gray[y1:y2, x1:x2]
        try:
            flow = cv2.calcOpticalFlowFarneback(prev_roi, curr_roi, None, 0.5, 3, 21, 3, 5, 1.2, 0)
            fx, fy = flow[..., 0], flow[..., 1]
            mag, ang = cv2.cartToPolar(fx, fy, angleInDegrees=False)
            mean_mag = float(np.mean(mag)) / max(ref_h, 1.0)
            p95_mag  = float(np.percentile(mag, 95)) / max(ref_h, 1.0)
            gx = cv2.Sobel(fx, cv2.CV_32F, 1, 0, ksize=3)
            gy = cv2.Sobel(fy, cv2.CV_32F, 0, 1, ksize=3)
            mbh = float(np.mean(np.abs(gx)) + np.mean(np.abs(gy))) / max(ref_h, 1.0)
            hist, _ = np.histogram(ang, bins=8, range=(0, 2*np.pi), weights=mag)
            hist = hist.astype(np.float32)
            hist = hist / (hist.sum() + 1e-6)
            entropy = float(-(hist * np.log(hist + 1e-6)).sum() / np.log(len(hist)))
            flow_score = min(1.0, 0.45 * (p95_mag / max(CONFIG.get("flow_motion_thr", 0.045), 1e-6)) +
                             0.35 * (mbh / max(CONFIG.get("flow_mbh_thr", 0.018), 1e-6)) +
                             0.20 * (entropy / max(CONFIG.get("flow_entropy_thr", 0.45), 1e-6)))
            return {"flow_mean": mean_mag, "flow_p95": p95_mag, "flow_mbh": mbh,
                    "flow_entropy": entropy, "flow_score": flow_score}
        except Exception:
            return {"flow_mean": 0.0, "flow_p95": 0.0, "flow_mbh": 0.0, "flow_entropy": 0.0, "flow_score": 0.0}

    def _svm_predict(self, feature_vec):
        if self.svm_model is None:
            return None
        try:
            arr = np.asarray(feature_vec, dtype=np.float32).reshape(1, -1)
            if hasattr(self.svm_model, "predict_proba"):
                return float(self.svm_model.predict_proba(arr)[0, 1])
            if hasattr(self.svm_model, "decision_function"):
                z = float(self.svm_model.decision_function(arr)[0])
                return 1.0 / (1.0 + np.exp(-z))
            return float(self.svm_model.predict(arr)[0])
        except Exception:
            return None

    def update(self, tracks, pose_map, frame_idx, frame=None, prev_frame=None):
        events = []
        centers = {}
        for tid, x1, y1, x2, y2 in tracks:
            sc = self.vel_tracker.get_center(tid)
            centers[tid] = sc if sc is not None else np.array([(x1+x2)/2.0, (y1+y2)/2.0], dtype=np.float32)

        prev_gray = curr_gray = None
        if frame is not None and prev_frame is not None and CONFIG.get("use_optical_flow", True):
            curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

        ids = sorted(centers.keys())
        alive_pairs = set()

        for a, b in combinations(ids, 2):
            if not (self._is_track_stable(a) and self._is_track_stable(b)):
                continue
            ta = next((t for t in tracks if t[0] == a), None)
            tb = next((t for t in tracks if t[0] == b), None)
            if ta is None or tb is None:
                continue

            ca, cb = centers[a], centers[b]
            dist_now = float(np.linalg.norm(ca - cb))
            ha = max(1.0, float(ta[4] - ta[2])); hb = max(1.0, float(tb[4] - tb[2]))
            ref_h = max(1.0, (ha + hb) * 0.5)
            dynamic_pair_limit = max(float(CONFIG["snatch_pair_max_dist_px"]), 3.2 * ref_h)
            key = (a, b); st = self.pair_state[key]; alive_pairs.add(key)

            if dist_now > dynamic_pair_limit:
                st["interaction_frames"] = 0
                raw, raw_prob = 0, 0.0
                metrics = {"distance_px": dist_now, "distance_norm": dist_now/ref_h, "phase": st["phase"],
                           "approach": 0.0, "approach_norm": 0.0, "escape_speed": 0.0, "escape_norm": 0.0,
                           "hand_speed": 0.0, "hand_norm": 0.0, "motion_cos": 0.0, "traj_divergence": 0.0,
                           "escape_asymmetry": 0.0, "flow_score": 0.0, "contact_recent": False}
            else:
                st["interaction_frames"] += 1
                pa = self.prev_centers.get(a, ca); pb = self.prev_centers.get(b, cb)
                va_raw = (ca - pa).astype(np.float32); vb_raw = (cb - pb).astype(np.float32)
                va, vb = self._smooth_velocity(key, va_raw, vb_raw)
                na, nb = float(np.linalg.norm(va)), float(np.linalg.norm(vb))
                motion_cos = float(np.dot(va, vb) / (na * nb)) if na > 1e-6 and nb > 1e-6 else 0.0

                prev_dist = float(np.linalg.norm(pa - pb))
                approach = max(0.0, prev_dist - dist_now)
                speed_a = float(np.linalg.norm(ca - pa)); speed_b = float(np.linalg.norm(cb - pb))
                escape_speed = max(speed_a, speed_b)
                escape_asymmetry = self._compute_escape_asymmetry(speed_a, speed_b)

                # =================================================================
                # [PHASE 3: ACTION ATTRIBUTE MODELLING]
                # Skeletal extraction of "Reaching" and "Grabbing" poses
                # =================================================================
                a_pose = pose_map.get(a); b_pose = pose_map.get(b)
                
                # Victim (a) and Suspect (b) Wrists (Indices 9, 10)
                a_lw = tuple(a_pose[9]) if a_pose is not None and len(a_pose) > 10 else None
                a_rw = tuple(a_pose[10]) if a_pose is not None and len(a_pose) > 10 else None
                b_lw = tuple(b_pose[9]) if b_pose is not None and len(b_pose) > 10 else None
                b_rw = tuple(b_pose[10]) if b_pose is not None and len(b_pose) > 10 else None
                
                # Suspect Shoulders (Indices 5, 6)
                b_ls = tuple(b_pose[5]) if b_pose is not None and len(b_pose) > 10 else None
                b_rs = tuple(b_pose[6]) if b_pose is not None and len(b_pose) > 10 else None

                # 1. Hand Speed (Retraction/Snatch speed)
                hand_speed = max(
                    self._wrist_speed(self.prev_wrists.get(a, {}).get("lw"), a_lw),
                    self._wrist_speed(self.prev_wrists.get(a, {}).get("rw"), a_rw),
                    self._wrist_speed(self.prev_wrists.get(b, {}).get("lw"), b_lw),
                    self._wrist_speed(self.prev_wrists.get(b, {}).get("rw"), b_rw),
                )

                # 2. Arm Extension (Distance from Suspect Shoulder to Suspect Wrist)
                arm_ext = 0.0
                if b_ls and b_lw and b_ls[2] > 0.4 and b_lw[2] > 0.4:
                    arm_ext = max(arm_ext, float(np.hypot(b_ls[0] - b_lw[0], b_ls[1] - b_lw[1])))
                if b_rs and b_rw and b_rs[2] > 0.4 and b_rw[2] > 0.4:
                    arm_ext = max(arm_ext, float(np.hypot(b_rs[0] - b_rw[0], b_rs[1] - b_rw[1])))
                
                # 3. Grab Proximity (Distance from Suspect Wrist to Victim Center)
                grab_prox = 999.0 # Default to far away
                if b_lw and b_lw[2] > 0.4:
                    grab_prox = min(grab_prox, float(np.hypot(ca[0] - b_lw[0], ca[1] - b_lw[1])))
                if b_rw and b_rw[2] > 0.4:
                    grab_prox = min(grab_prox, float(np.hypot(ca[0] - b_rw[0], ca[1] - b_rw[1])))
                
                # Normalize the new skeletal metrics
                arm_ext_norm = arm_ext / ref_h
                grab_prox_norm = grab_prox / ref_h if grab_prox < 999.0 else 1.0
                # =================================================================

                dist_norm = dist_now / ref_h
                hand_norm = hand_speed / ref_h
                approach_norm = approach / ref_h
                escape_norm = escape_speed / ref_h
                close_contact = dist_norm <= CONFIG["snatch_close_dist_norm"]
                if close_contact:
                    st["last_contact_frame"] = frame_idx
                    st["contact_frames"] += 1
                contact_recent = (frame_idx - st["last_contact_frame"]) <= CONFIG.get("contact_memory_frames", 18)

                flow = self._flow_features(prev_gray, curr_gray, ta, tb, ref_h)
                proximity_score = 1.0 - min(1.0, dist_norm / max(CONFIG["snatch_close_dist_norm"], 1e-6))
                approach_score = min(1.0, approach_norm / max(CONFIG["snatch_approach_thr"], 1e-6))
                escape_score = min(1.0, escape_norm / max(CONFIG["snatch_escape_thr"], 1e-6))
                asym_score = min(1.0, escape_asymmetry / max(CONFIG.get("snatch_escape_asymmetry_thr", 0.16), 1e-6))
                hand_score = min(1.0, hand_norm / max(CONFIG["snatch_hand_speed_thr"], 1e-6)) if hand_speed > 0 else 0.0
                flow_score = flow["flow_score"]

                raw_prob = (0.10 * proximity_score + 0.14 * approach_score + 0.25 * escape_score +
                            0.20 * asym_score + 0.21 * flow_score + 0.10 * hand_score)
                same_direction = na > 1.5 and nb > 1.5 and motion_cos >= CONFIG["snatch_same_direction_cosine"]
                if same_direction and asym_score < 0.55 and flow_score < 0.45:
                    raw_prob *= CONFIG.get("same_direction_penalty", 0.55)

                feature_vec = [dist_norm, approach_norm, escape_norm, hand_norm, motion_cos,
                               escape_asymmetry, flow["flow_mean"], flow["flow_p95"],
                               flow["flow_mbh"], flow["flow_entropy"], flow_score,
                               float(contact_recent), float(st["contact_frames"]),
                               arm_ext_norm, grab_prox_norm] # <-- Added Skeletal Attributes
                svm_prob = self._svm_predict(feature_vec)
                
                if svm_prob is not None:
                    raw_prob = 0.45 * raw_prob + 0.55 * svm_prob

                phase_age = frame_idx - st["phase_frame"]
                if st["phase"] == self.IDLE:
                    if approach_score > 0.45 or (close_contact and flow_score > 0.25):
                        st["phase"] = self.APPROACH
                        st["phase_frame"] = frame_idx
                        st["approach_frames"] = 1
                elif st["phase"] == self.APPROACH:
                    if approach_score > 0.35:
                        st["approach_frames"] += 1
                    if close_contact:
                        st["phase"] = self.CONTACT
                        st["phase_frame"] = frame_idx
                    elif phase_age > 45:
                        st["phase"] = self.IDLE
                        st["approach_frames"] = 0
                        st["contact_frames"] = 0
                elif st["phase"] == self.CONTACT:
                    escape_evidence = (escape_score > 0.70 and asym_score > 0.60) or (escape_score > 0.55 and flow_score > 0.55)
                    if contact_recent and escape_evidence:
                        st["phase"] = self.ESCAPE
                        st["phase_frame"] = frame_idx
                        st["escape_confirmed"] = True
                    elif phase_age > 35 and not contact_recent:
                        st["phase"] = self.IDLE
                        st["approach_frames"] = 0
                        st["contact_frames"] = 0
                elif st["phase"] == self.ESCAPE:
                    if phase_age > 25:
                        st["phase"] = self.IDLE
                        st["escape_confirmed"] = False
                        st["contact_frames"] = 0

                traj_divergence = 0.0
                if self.track_history is not None:
                    ta_traj = self.track_history.get_trajectory(a)
                    tb_traj = self.track_history.get_trajectory(b)
                    w = CONFIG.get("traj_divergence_window", 10)
                    if len(ta_traj) >= 2 and len(tb_traj) >= 2:
                        n = min(len(ta_traj), len(tb_traj), w)
                        d_start = float(np.hypot(ta_traj[-n][0]-tb_traj[-n][0], ta_traj[-n][1]-tb_traj[-n][1]))
                        d_end = float(np.hypot(ta_traj[-1][0]-tb_traj[-1][0], ta_traj[-1][1]-tb_traj[-1][1]))
                        traj_divergence = (d_end - d_start) / ref_h

                phase_gate = st["escape_confirmed"] or st["phase"] in (self.ESCAPE, self.ALERT)
                strong_escape = escape_score > 0.70 and asym_score > 0.55
                strong_flow = flow_score > 0.68 and escape_score > 0.45
                raw = int(
                    phase_gate and contact_recent and
                    st["interaction_frames"] >= CONFIG["snatch_min_interaction_frames"] and
                    (strong_escape or strong_flow or (svm_prob is not None and svm_prob >= 0.55)) and
                    raw_prob >= self._eff_prob_threshold()
                )

                metrics = {
                    "distance_px": dist_now, "distance_norm": dist_norm,
                    "approach": approach, "approach_norm": approach_norm,
                    "escape_speed": escape_speed, "escape_norm": escape_norm,
                    "hand_speed": hand_speed, "hand_norm": hand_norm,
                    "motion_cos": motion_cos, "traj_divergence": traj_divergence,
                    "escape_asymmetry": escape_asymmetry, "phase": st["phase"],
                    "approach_frames": st["approach_frames"], "contact_frames": st["contact_frames"],
                    "contact_recent": contact_recent, "raw_prob": raw_prob,
                    "flow_score": flow_score, "flow_mean": flow["flow_mean"],
                    "flow_p95": flow["flow_p95"], "flow_mbh": flow["flow_mbh"],
                    "flow_entropy": flow["flow_entropy"], "svm_prob": svm_prob,
                }

                
                # =================================================================
                # [DATA COLLECTION WIRETAP] 
                # Save physics data when a pair escapes, regardless of alarms
                # =================================================================
                if contact_recent and st["phase"] in (self.ESCAPE, self.ALERT):
                    last_harvest = st.get("last_harvest_frame", -9999)
                    if frame_idx - last_harvest > 30: # 1-second cooldown
                        self.harvested_features.append({
                            "category": self.category_name,
                            "dist_norm": dist_norm,
                            "approach_norm": approach_norm,
                            "escape_norm": escape_norm,
                            "hand_norm": hand_norm,
                            "motion_cos": motion_cos,
                            "escape_asymmetry": escape_asymmetry,
                            "flow_mean": flow["flow_mean"],
                            "flow_p95": flow["flow_p95"],
                            "flow_mbh": flow["flow_mbh"],
                            "flow_entropy": flow["flow_entropy"],
                            "flow_score": flow_score,
                            "contact_recent": float(contact_recent),
                            "contact_frames": float(st["contact_frames"]),
                            "arm_extension": arm_ext_norm,      # <-- Phase 3 Metric
                            "grab_proximity": grab_prox_norm    # <-- Phase 3 Metric
                        })
                        st["last_harvest_frame"] = frame_idx
                # =================================================================
                

                if CONFIG.get("record_pair_sequences"):
                    st["seq"].append(feature_vec + [raw_prob, float(raw)])

            st["history"].append(raw)
            positives = sum(st["history"])
            st["last_metrics"] = metrics

            if (not st["active"]) and positives >= CONFIG["snatch_n_on"]:
                st["active"] = True
                st["phase"] = self.ALERT
                st["phase_frame"] = frame_idx
            elif st["active"] and positives <= CONFIG["snatch_n_off"]:
                st["active"] = False

            # [FIX] Cooldown Gate: Prevent the system from logging 30 times a second!
            if st["active"]:
                last_frame = self.last_alert_frame.get(key, -9999)
                
                # Only log the event if 30 frames (1 seconds) have passed since the last alert for this pair
                if frame_idx - last_frame > 30:
                    ev_conf = round(max(metrics.get("raw_prob", 0.0), positives / max(len(st["history"]), 1)), 3)
                    events.append({
                        "event_type": "SNATCHING_DETECTED",
                        "frame": frame_idx,
                        "victim_id": a,
                        "suspect_id": b,
                        "confidence": ev_conf,
                        "distance_px": round(metrics.get("distance_px", dist_now), 1),
                        "distance_norm": round(metrics.get("distance_norm", 0.0), 3),
                        "approach_speed": round(metrics.get("approach", 0.0), 2),
                        "approach_norm": round(metrics.get("approach_norm", 0.0), 3),
                        "escape_speed": round(metrics.get("escape_speed", 0.0), 2),
                        "escape_norm": round(metrics.get("escape_norm", 0.0), 3),
                        "hand_speed": round(metrics.get("hand_speed", 0.0), 2),
                        "hand_norm": round(metrics.get("hand_norm", 0.0), 3),
                        "motion_cos": round(metrics.get("motion_cos", 0.0), 3),
                        "traj_divergence": round(metrics.get("traj_divergence", 0.0), 3),
                        "escape_asymmetry": round(metrics.get("escape_asymmetry", 0.0), 3),
                        "flow_score": round(metrics.get("flow_score", 0.0), 3),
                        "flow_mbh": round(metrics.get("flow_mbh", 0.0), 3),
                        "phase": metrics.get("phase", self.IDLE),
                        "approach_frames": metrics.get("approach_frames", 0),
                        "contact_frames": metrics.get("contact_frames", 0),
                        "svm_prob": metrics.get("svm_prob", 0.0)
                    })
                    
                    # Update the cooldown timer so it waits before logging again
                    self.last_alert_frame[key] = frame_idx
        grace = CONFIG.get("pair_grace_frames", 30)
        for k in list(self.pair_state.keys()):
            if k not in alive_pairs:
                self.pair_state[k]["missing_frames"] = self.pair_state[k].get("missing_frames", 0) + 1
                if self.pair_state[k]["missing_frames"] > grace:
                    del self.pair_state[k]
            else:
                self.pair_state[k]["missing_frames"] = 0

        new_wr = {}
        for tid in ids:
            pose = pose_map.get(tid)
            if pose is not None and len(pose) > 10:
                new_wr[tid] = {"lw": tuple(pose[9]), "rw": tuple(pose[10])}
        self.prev_wrists = new_wr
        self.prev_centers = centers
        return events


In [11]:
# Cell 9 - draw_frame_clean (center-point calculation hardened)
def draw_frame_clean(frame, tracks, items, pose_map, violations, zone_info, pixel_zones,
                     frame_idx, fps, cooldown_frac=1.0, smoothed_conf=0.0,
                     frame_was_enhanced=False):
    ALERT_COLOR = CONFIG["alert_color"]
    SAFE_COLOR  = CONFIG["safe_color"]
    WHITE       = (255, 255, 255)
    BLACK       = (0,   0,   0)
    FONT        = cv2.FONT_HERSHEY_SIMPLEX
    ITEM_COLOR  = (255, 200, 0)
    
    h, w = frame.shape[:2]

    for z in pixel_zones:
        cv2.polylines(frame, [z["polygon_np"]], isClosed=True,
                      color=(200, 200, 200), thickness=1)
        px, py = z["polygon_px"][0]
        cv2.putText(frame, z["name"], (px+4, py+18), FONT, 0.45, (200,200,200), 1)

    # 2. DRAW SKELETONS (NEW INTEGRATION)
    # Mapping for COCO/YOLO-Pose format
    skeleton_lines = [(5, 7), (7, 9), (6, 8), (8, 10)] # Shoulders to wrists
    for tid, kp in pose_map.items():
        if kp is not None:
            # Draw joints
            for i in range(5, 11): # Only draw upper body joints
                x, y, conf = kp[i]
                if conf > 0.3:
                    cv2.circle(frame, (int(x), int(y)), 4, (255, 0, 255), -1)
            # Draw limbs
            for p1, p2 in skeleton_lines:
                if kp[p1, 2] > 0.3 and kp[p2, 2] > 0.3:
                    cv2.line(frame, (int(kp[p1, 0]), int(kp[p1, 1])), 
                                     (int(kp[p2, 0]), int(kp[p2, 1])), (0, 255, 255), 1)

    # 3. DRAW BOXES & DEBUG TEXT (Existing + NEW SVM INFO)
    alert_ids = set()
    for v in violations:
        if v.get("event_type") == "SNATCHING_DETECTED":
            alert_ids.add(v.get("victim_id")); alert_ids.add(v.get("suspect_id"))
        elif v.get("event_type") == "ZONE_VIOLATION":
            alert_ids.add(v.get("track_id"))

    track_map = {t[0]: t for t in tracks}

    for tid, x1, y1, x2, y2 in tracks:
        is_alert = (tid in alert_ids) or zone_info.get(tid, {}).get("is_violating", False)
        color = ALERT_COLOR if is_alert else SAFE_COLOR
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        
        # New Debug Text: Show SVM Prob if available
        label = f"#{tid} {'ALERT' if is_alert else 'OK'}"
        cv2.putText(frame, label, (x1+2, y1-10), FONT, 0.45, WHITE, 1)

    # 4. DRAW VIOLATION LINES (Existing)
    for v in violations:
        if v.get("event_type") == "SNATCHING_DETECTED":
            aid, bid = v["victim_id"], v["suspect_id"]
            ta, tb = track_map.get(aid), track_map.get(bid)
            if ta and tb:
                vc = ((ta[1]+ta[3])//2, (ta[2]+ta[4])//2)
                sc = ((tb[1]+tb[3])//2, (tb[2]+tb[4])//2)
                cv2.line(frame, vc, sc, ALERT_COLOR, 2)
                
               # Show SVM info directly on the alert line safely
                svm_p = v.get("svm_prob")
                svm_text = f"SVM:{svm_p:.2f}" if svm_p is not None else "SVM: --"
                
                cv2.putText(frame, svm_text, ((vc[0]+sc[0])//2, (vc[1]+sc[1])//2), 
                            FONT, 0.4, (0, 255, 255), 1)
    for itm in items:
        iid, cls, ix1, iy1, ix2, iy2 = itm
        cv2.rectangle(frame, (ix1, iy1), (ix2, iy2), ITEM_COLOR, 2)
        # Optional: Label the item
        cv2.putText(frame, f"Item", (ix1, iy1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, ITEM_COLOR, 1)
        
    if violations:
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 48), (20, 0, 100), -1)
        cv2.addWeighted(overlay, 0.80, frame, 0.20, 0, frame)
        cv2.line(frame, (0, 48), (w, 48), ALERT_COLOR, 2)
        n_snatch = sum(1 for v in violations if v.get("event_type")=="SNATCHING_DETECTED")
        n_zone   = sum(1 for v in violations if v.get("event_type")=="ZONE_VIOLATION")
        parts = []
        if n_snatch: parts.append(f"SNATCH x{n_snatch}")
        if n_zone:   parts.append(f"ZONE x{n_zone}")
        alert_txt = "  !  " + "  |  ".join(parts) + f"  |  conf:{smoothed_conf:.0%}"
        cv2.putText(frame, alert_txt, (8, 33), FONT, 0.60, WHITE, 2, cv2.LINE_AA)
        fill_w = int(w * max(0.0, 1.0 - cooldown_frac))
        cv2.rectangle(frame, (0, 44), (w, 48), (40, 0, 60), -1)
        if fill_w > 0:
            cv2.rectangle(frame, (0, 44), (fill_w, 48), ALERT_COLOR, -1)

    info = f"F:{frame_idx}  FPS:{fps:.1f}"
    if frame_was_enhanced:
        info += "  [VIS-ENH]"
    cv2.putText(frame, info, (6, h-8), FONT, 0.38, (180, 180, 180), 1)
    return frame


In [12]:
# Cell 10 - Accuracy helper  
def compute_per_video_stability(frame_detections, video_name, fps, total_frames, total_violations):
    """These are DETECTION-STABILITY metrics, NOT alert-correctness metrics.
    Correctness requires ground truth -> see evaluate_alerts()."""
    if not frame_detections: return {}
    df  = pd.DataFrame(frame_detections)
    fwd = (df["n_detections"] > 0).sum()
    dr  = fwd / max(1, len(df))
    ac  = float(df["avg_conf"].mean()) if "avg_conf" in df.columns else 0.0
    ap  = float(df["n_detections"].mean())
    sp  = float(df["n_detections"].std())
    cons= 1.0 - min(1.0, sp / max(1, ap))
    dm  = total_frames / max(1, fps) / 60.0
    vr  = total_violations / max(0.001, dm)
    si  = 0.50*dr + 0.30*ac + 0.20*cons
    return {
        "video":            video_name,
        "total_frames":     total_frames,
        "duration_min":     round(dm, 2),
        "frames_with_det":  int(fwd),
        "detection_rate":   round(dr, 4),
        "avg_conf":         round(ac, 4),
        "avg_persons":      round(ap, 2),
        "consistency":      round(cons, 4),
        "total_violations": total_violations,
        "violation_rate":   round(vr, 2),
        "stability_index":  round(si, 4),   
    }


In [13]:
# Cell 10b ------------------------------
def evaluate_alerts(alert_log, ground_truth_csv, temporal_iou_thr=0.2):
    """
    GT CSV columns: video, event_start_frame, event_end_frame, type
    Matches predicted alert spans to GT events by temporal IoU and reports
    precision / recall / F1 + mean time-to-alert (latency, in frames).
    """
    if not os.path.exists(ground_truth_csv):
        return {"error": f"ground truth not found: {ground_truth_csv}"}
    gt = pd.read_csv(ground_truth_csv)
    alerts = pd.DataFrame(alert_log)
    if alerts.empty:
        return {"precision":0.0,"recall":0.0,"f1":0.0,"avg_latency_frames":None,
                "matched":0,"n_alerts":0,"n_gt":len(gt)}

    # collapse consecutive alert frames per (video, event_type) into spans
    spans = []
    for (vid, etype), g in alerts.groupby(["video","event_type"]):
        fr = sorted(g["frame"].tolist())
        s = fr[0]; prev = fr[0]
        for f in fr[1:]:
            if f - prev > 15:           # gap -> new span
                spans.append((vid, etype, s, prev)); s = f
            prev = f
        spans.append((vid, etype, s, prev))

    def tiou(a0,a1,b0,b1):
        inter = max(0, min(a1,b1) - max(a0,b0))
        union = max(a1,b1) - min(a0,b0)
        return inter / max(union, 1e-9)

    matched_gt, latencies, used = 0, [], set()
    for _, r in gt.iterrows():
        best, best_span = 0.0, None
        for i, (vid, etype, s, e) in enumerate(spans):
            if vid != r["video"]: continue
            t = tiou(int(r["event_start_frame"]), int(r["event_end_frame"]), s, e)
            if t > best: best, best_span = t, (i, s)
        if best >= temporal_iou_thr and best_span and best_span[0] not in used:
            matched_gt += 1; used.add(best_span[0])
            latencies.append(best_span[1] - int(r["event_start_frame"]))

    n_alerts = len(spans); n_gt = len(gt)
    precision = matched_gt / max(1, n_alerts)
    recall    = matched_gt / max(1, n_gt)
    f1 = 2*precision*recall / max(1e-9, precision+recall)
    return {
        "precision": round(precision,4), "recall": round(recall,4), "f1": round(f1,4),
        "avg_latency_frames": round(float(np.mean(latencies)),1) if latencies else None,
        "matched": matched_gt, "n_alerts": n_alerts, "n_gt": n_gt,
    }


In [14]:
# Cell 11 - Load models + safe warm-up
import os
import torch
import numpy as np
from ultralytics import YOLO

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

phase_log("PHASE 6 - Detection Engine")

def select_safe_device():
    """
    torch.cuda.is_available() can be True even when the Kaggle GPU cannot run
    the installed CUDA kernels. This test actually runs a tiny CUDA operation.
    """
    if not torch.cuda.is_available():
        return "cpu", False

    try:
        x = torch.zeros((1, 3, 32, 32), device="cuda")
        _ = x + 1
        torch.cuda.synchronize()
        return 0, True
    except Exception as e:
        print("CUDA is visible but unusable for this runtime.")
        print("Falling back to CPU.")
        print("CUDA test error:", repr(e))
        return "cpu", False


SAFE_DEVICE, CUDA_OK = select_safe_device()

CONFIG["device"] = SAFE_DEVICE
CONFIG["half"] = False if SAFE_DEVICE == "cpu" else bool(CONFIG.get("half", False))

print("Selected device:", CONFIG["device"])
print("Half precision:", CONFIG["half"])

detector_model = YOLO(BEST_MODEL_PATH)

DETECTOR_NAMES = getattr(detector_model, "names", {}) or {}
DETECTOR_NC = len(DETECTOR_NAMES) if hasattr(DETECTOR_NAMES, "__len__") else 0
DETECTOR_CLASS_IDS = [0]

print("Detector class names:", DETECTOR_NAMES)

if DETECTOR_NC == 1 and str(DETECTOR_NAMES.get(0, "")).lower() == "person":
    print("Checkpoint check: one-class person detector.")
    print("Snatching must be inferred from tracking + motion/action logic.")
else:
    print("Warning: detector class names are not the expected {0: 'person'}.")
    print("Still using class 0 as person.")

POSE_MODEL = "yolov8n-pose.pt"
pose_model = YOLO(POSE_MODEL)

enhancer = FrameEnhancer(
    clip_limit=CONFIG["clahe_clip_limit"],
    tile_grid=CONFIG["clahe_tile_grid"],
    brightness_thr=CONFIG["clahe_brightness_thr"],
    contrast_thr=CONFIG["clahe_fog_contrast_thr"],
)

TRACKER_CFG = ensure_tracker_cfg(CONFIG["tracker_cfg_path"])

print("Models loaded")
print("   Detector :", BEST_MODEL_PATH)
print("   Task     :", detector_model.task)
print("   Classes  :", DETECTOR_NAMES)
print("   Pose     :", POSE_MODEL)
print("   Tracker  :", TRACKER_CFG, f"(with_reid={CONFIG['tracker_with_reid']})")
print("   Enhancer : CLAHE adaptive")

# Safe warm-up. If GPU warm-up fails, retry on CPU.
dummy = np.zeros((CONFIG["infer_imgsz"], CONFIG["infer_imgsz"], 3), dtype=np.uint8)

try:
    detector_model(
        dummy,
        classes=DETECTOR_CLASS_IDS,
        conf=CONFIG["confidence"],
        verbose=False,
        imgsz=CONFIG["infer_imgsz"],
        device=CONFIG["device"],
    )

    pose_model(
        dummy,
        classes=[0],
        conf=CONFIG["confidence"],
        verbose=False,
        imgsz=CONFIG["infer_imgsz"],
        device=CONFIG["device"],
    )

    print("Warm-up done")

except Exception as e:
    print("Warm-up failed on selected device:", repr(e))
    print("Retrying on CPU...")

    CONFIG["device"] = "cpu"
    CONFIG["half"] = False

    detector_model(
        dummy,
        classes=DETECTOR_CLASS_IDS,
        conf=CONFIG["confidence"],
        verbose=False,
        imgsz=CONFIG["infer_imgsz"],
        device="cpu",
    )

    pose_model(
        dummy,
        classes=[0],
        conf=CONFIG["confidence"],
        verbose=False,
        imgsz=CONFIG["infer_imgsz"],
        device="cpu",
    )

    print("CPU warm-up done")


----------------------------------------------------------------------
    PHASE 6 - Detection Engine  |  Elapsed: 0.0 min
----------------------------------------------------------------------
CUDA is visible but unusable for this runtime.
Falling back to CPU.
CUDA test error: AcceleratorError("CUDA error: no kernel image is available for execution on the device\nSearch for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n")
Selected device: cpu
Half precision: False
Detector class names: {0: 'person'}
Checkpoint check: one-class person detector.
Snatching must be inferred from tracking + motion/action logic.
Models loaded
   Detector : /kaggle/input/datasets/sh

In [15]:
# Cell 12 - Detection function (all fixes integrated)
def run_detection(video_path, output_video_path=None, max_frames=None, category_name=None):
    print(f"\n {Path(video_path).name}")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("   Cannot open"); return [], [], {}, None

    if hasattr(detector_model, "predictor") and detector_model.predictor is not None:
        detector_model.predictor.trackers = []

    fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    limit  = min(total, max_frames) if max_frames else total

    writer = None
    if output_video_path:
        mp4_path = str(Path(output_video_path).with_suffix(".mp4"))
        writer   = cv2.VideoWriter(mp4_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

    # [FIX 3] per-frame track log (guard when no output path)
    track_log_fh = None
    if CONFIG["track_log_enabled"] and output_video_path:
        track_log_path = str(Path(output_video_path).with_suffix(".tracks.jsonl.gz"))
        track_log_fh   = gzip.open(track_log_path, "wt")

    # Per-video state
    box_smoother  = BoxSmoother(win=CONFIG["box_smooth_len"])      # [FIX 2]
    track_history = TrackHistoryManager(                          # [P3]
        history_len=CONFIG["track_history_len"],
        min_reidentify_frames=CONFIG["track_history_min_frames"],
    )
    # [REID] occlusion recovery uses track_history for motion-predicted matching,
    # so it is built AFTER track_history.
    occ_recovery  = OcclusionRecovery(                            # [P2c]
        iou_thr=CONFIG["occ_iou_thr"],
        max_gap_frames=CONFIG["occ_max_gap_frames"],
        size_ratio_tol=CONFIG["occ_size_ratio_tol"],
        motion_gate=CONFIG["occ_motion_gate"],
        max_pred_dist_norm=CONFIG["occ_max_pred_dist_norm"],
        ambiguity_margin=CONFIG["occ_ambiguity_margin"],
        score_thr=CONFIG["occ_score_thr"],
        history=track_history,
    ) if CONFIG.get("occlusion_recovery") else None
    vel_tracker   = VelocitySmoothedTracker()
    ensure_zone_config(CONFIG["zone_config_path"])
    pixel_zones   = load_polygon_zones(CONFIG["zone_config_path"], width, height,
                                       video_path=video_path, category_name=category_name)
    zone_branch   = ZoneViolationBranch(pixel_zones,
                                        min_frames=CONFIG["zone_violation_frames"],
                                        hold_frames=CONFIG.get("alert_hold_frames", 60))
    snatch_branch = SnatchingBranch(vel_tracker, track_history=track_history, category_name=category_name)
    persistence   = AlertPersistenceManager(hold_frames=CONFIG.get("alert_hold_frames", 60))

    print(f"  Zones: {len(pixel_zones)} | {[z['name'] for z in pixel_zones]}")

    alert_log, snapshot_frames, frame_detections = [], [], []
    frame_num, total_viols = 0, 0
    enhanced_count, multiscale_count = 0, 0
    # [FIX 1] CLAHE burst state
    clahe_active, clahe_countdown = False, 0
    t0 = time.time()
    pbar = tqdm(total=limit, desc="  Detecting", unit="fr")
    prev_infer_frame = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or (max_frames and frame_num >= max_frames): break
        frame_num += 1

        # -- [FIX 1] CLAHE in bursts: re-check lighting only every N frames --
        frame_was_enhanced = False
        infer_frame = frame
        if CONFIG["clahe_enabled"]:
            if frame_num % CONFIG["clahe_eval_interval"] == 1:
                if enhancer.needs_enhancement(frame):
                    clahe_active = True
                    clahe_countdown = CONFIG["clahe_burst_len"]
            if clahe_active:
                infer_frame = enhancer.enhance_only(frame)   # no redundant re-check
                frame_was_enhanced = True
                enhanced_count += 1
                clahe_countdown -= 1
                if clahe_countdown <= 0:
                    clahe_active = False

                # The attached best_model checkpoint is trained as nc=1 / names={0: 'person'}.
        # Track only people here. Snatching is inferred from motion/action attributes after tracking.
        results = detector_model.track(
            infer_frame, persist=True, tracker=TRACKER_CFG,
            classes=DETECTOR_CLASS_IDS, conf=CONFIG["confidence"], iou=CONFIG["iou_thresh"],
            imgsz=CONFIG["infer_imgsz"], device=CONFIG["device"], verbose=False,
        )[0]

        tracks, items, avg_conf_frame = [], [], 0.0
        if results.boxes is not None and len(results.boxes):
            boxes_np = results.boxes.xyxy.cpu().numpy()
            confs_np = results.boxes.conf.cpu().numpy()
            cls_np = results.boxes.cls.cpu().numpy().astype(int) if results.boxes.cls is not None else np.zeros(len(boxes_np), dtype=int)
            ids_np = results.boxes.id.cpu().numpy().astype(int) if results.boxes.id is not None else np.arange(len(boxes_np))

            person_confs = []
            for box, conf, cls_id, tid in zip(boxes_np, confs_np, cls_np, ids_np):
                x1, y1, x2, y2 = map(int, box)
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(width, x2), min(height, y2)
                if x2 <= x1 or y2 <= y1:
                    continue

                if int(cls_id) == 0:
                    tracks.append((int(tid), x1, y1, x2, y2))
                    person_confs.append(float(conf))
                # Non-person classes are ignored because the supplied checkpoint is person-only.

            avg_conf_frame = float(np.mean(person_confs)) if person_confs else 0.0

        # -- [FIX 1] Multi-scale ROI zoom (<= ms_max_roi extra passes) ------
        # [FIX 2] extra detections are NOT given synthetic persistent IDs, so
        # they never pollute frame-to-frame tracking or snatch pairing. They are
        # only recorded (raw) for logging/recall accounting.
        extra_detections = []
        if CONFIG["use_multi_scale"] and should_use_multiscale(infer_frame, tracks, CONFIG):
            multiscale_count += 1
            extra_boxes = run_multiscale_extra(detector_model, infer_frame, tracks, CONFIG)
            existing = [[t[1],t[2],t[3],t[4]] for t in tracks]
            for eb in extra_boxes:
                ex1,ey1,ex2,ey2,ec = eb
                # [FIX 6] IoU gate (not any-pixel overlap)
                if all(_iou_xyxy([ex1,ey1,ex2,ey2], ex) < CONFIG["ms_merge_iou"]
                       for ex in existing):
                    extra_detections.append([int(ex1),int(ey1),int(ex2),int(ey2),round(ec,3)])

        # [P2c] occlusion ID-recovery remap BEFORE any per-id state is keyed,
        # so velocity/history/pair state stay attached to the recovered id.
        if occ_recovery is not None:
            tracks = occ_recovery.update(tracks, frame_num)

        # [FIX 2] per-ID smoothing (no cross-id interpolation / ghost boxes)
        tracks = box_smoother.update(tracks)
        tracks = vel_tracker.update(tracks)
        track_history.update(tracks, frame_num)   # [P3] persistent trajectory

        # -- [FIX 1] Pose only when a plausible interaction pair exists -----
        pose_map = {}
        run_pose = True
        if CONFIG["pose_only_if_pair"]:
            run_pose = False
            if len(tracks) >= 2:
                ctrs = [((t[1]+t[3])/2.0, (t[2]+t[4])/2.0) for t in tracks]
                for i in range(len(ctrs)):
                    for j in range(i+1, len(ctrs)):
                        if np.hypot(ctrs[i][0]-ctrs[j][0], ctrs[i][1]-ctrs[j][1]) \
                           < CONFIG["snatch_pair_max_dist_px"]:
                            run_pose = True; break
                    if run_pose: break

        if run_pose:
            pose_result = pose_model(
                infer_frame, classes=[0], conf=CONFIG["confidence"],
                verbose=False, imgsz=CONFIG["infer_imgsz"], device=CONFIG["device"],
            )[0]
            if (pose_result.boxes is not None and pose_result.keypoints is not None
                    and len(pose_result.boxes) and len(tracks)):
                p_boxes   = pose_result.boxes.xyxy.cpu().numpy()
                p_kpts    = pose_result.keypoints.data.cpu().numpy()
                t_ids     = [t[0] for t in tracks]
                t_boxes   = np.array([[t[1],t[2],t[3],t[4]] for t in tracks], dtype=np.float32)
                t_centers = np.array([[(t[1]+t[3])/2.,(t[2]+t[4])/2.] for t in tracks], dtype=np.float32)

                for pb, pk in zip(p_boxes, p_kpts):
                    ious = np.array([_iou_xyxy(pb, tb) for tb in t_boxes], dtype=np.float32)
                    best = int(np.argmax(ious))
                    if float(ious[best]) >= 0.10:
                        pose_map[t_ids[best]] = pk
                    else:
                        pc  = np.array([(pb[0]+pb[2])/2.,(pb[1]+pb[3])/2.], dtype=np.float32)
                        idx = int(np.argmin(np.linalg.norm(t_centers - pc, axis=1)))
                        pose_map[t_ids[idx]] = pk

        # -- Branch updates -------------------------------------------------
        zone_events, zone_info = zone_branch.update(tracks, frame_num)
        snatch_events          = snatch_branch.update(tracks, pose_map, frame_num, frame=infer_frame, prev_frame=prev_infer_frame)
        violations             = zone_events + snatch_events
        display_violations, cooldown_frac = persistence.update(violations)

        frame_detections.append({
            "frame":        frame_num,
            "n_detections": len(tracks),
            "avg_conf":     avg_conf_frame,
            "n_violations": len(violations),
            "enhanced":     frame_was_enhanced,
        })

        if track_log_fh is not None:
            entry = {"frame": frame_num, "enhanced": frame_was_enhanced,
                     "tracks": [], "extra_detections": extra_detections,
                     # [REID] traceable remap decisions for this frame
                     "reid_decisions": (occ_recovery.frame_decisions
                                        if occ_recovery is not None else [])}
            for tid, x1, y1, x2, y2 in tracks:
                c = vel_tracker.get_center(tid)
                v = vel_tracker.velocities.get(tid)
                rec = {
                    "id": int(tid),
                    "bbox": [int(x1), int(y1), int(x2), int(y2)],
                    "center": [int((x1+x2)//2), int((y1+y2)//2)],
                    "velocity": [float(v[0]), float(v[1])] if v is not None else [0.0, 0.0],
                    "zone": zone_info.get(tid, {}).get("in_zones", []),
                    "violating": zone_info.get(tid, {}).get("is_violating", False),
                }
                if tid in pose_map:
                    rec["keypoints"] = pose_map[tid].tolist()
                entry["tracks"].append(rec)
            track_log_fh.write(json.dumps(entry) + "\n")
            if frame_num % CONFIG["track_log_flush_every"] == 0:
                track_log_fh.flush()

        if violations:
            total_viols += len(violations)
            for v in violations:
                _vs = zone_info.get(v.get("victim_id", v.get("track_id")), {"in_zones":[]})
                _ss = zone_info.get(v.get("suspect_id", -1), {"in_zones":[]})
                alert_log.append({
                    "frame":         frame_num,
                    "time_sec":      round(frame_num/fps, 2),
                    "event_type":    v.get("event_type","UNKNOWN"),
                    "victim_id":     v.get("victim_id", v.get("track_id")),
                    "suspect_id":    v.get("suspect_id"),
                    "victim_zone":   ",".join(_vs.get("in_zones",[])) or "CLEAR",
                    "suspect_zone":  ",".join(_ss.get("in_zones",[])) or "CLEAR",
                    "same_zone":     set(_vs.get("in_zones",[])) == set(_ss.get("in_zones",[])),
                    "distance_px":   v.get("distance_px"),
                    "zone_id":       v.get("zone_id"),
                    "zone_name":     v.get("zone_name"),
                    "violation_conf":v.get("confidence", 0.0),
                    "violation_flag":True,
                })
            if len(snapshot_frames) < 5:
                snapshot_frames.append((frame_num, frame.copy(), violations))

        annotated = draw_frame_clean(
            frame, tracks, items, pose_map, display_violations, zone_info, pixel_zones,
            frame_num, fps, cooldown_frac=cooldown_frac,
            smoothed_conf=persistence.smoothed_conf,
            frame_was_enhanced=frame_was_enhanced,
        )

        if writer: writer.write(annotated)
        prev_infer_frame = infer_frame.copy()
        pbar.update(1)

    pbar.close()
    cap.release()
    if writer: writer.release()
    if track_log_fh is not None: track_log_fh.close()

    proc_fps = frame_num / max(time.time()-t0, 1e-9)
    enh_pct  = 100.0 * enhanced_count   / max(1, frame_num)
    ms_pct   = 100.0 * multiscale_count / max(1, frame_num)
    print(f"  {frame_num} frames | {total_viols} violations | {proc_fps:.1f} fps")
    print(f"  CLAHE enhanced: {enhanced_count} frames ({enh_pct:.1f}%)")
    print(f"  Multi-scale:    {multiscale_count} frames ({ms_pct:.1f}%)")

    vid_acc = compute_per_video_stability(frame_detections, Path(video_path).name,
                                          fps, frame_num, total_viols)
    vid_acc["enhanced_frames_pct"]   = round(enh_pct, 2)
    vid_acc["multiscale_frames_pct"] = round(ms_pct, 2)
    print(f"  DetRate:{vid_acc.get('detection_rate',0):.1%}  "
          f"Conf:{vid_acc.get('avg_conf',0):.3f}  "
          f"Stability:{vid_acc.get('stability_index',0):.3f}")

    mp4_out = str(Path(output_video_path).with_suffix(".mp4")) if output_video_path else None
    return alert_log, snapshot_frames, vid_acc, mp4_out,snatch_branch.harvested_features


In [16]:
# Cell 13 - Main detection loop (with Stratified Random Sampling)
import random
phase_log("PHASE 7 - Video Detection Loop")

all_alert_logs      = []
all_snapshot_frames = []
all_video_accuracy  = []
generated_videos    = []
MASTER_DATASET      = []

# or set to None for a completely random draw every run!
EVALUATION_SEED     = 42 
rng = random.Random(EVALUATION_SEED)

for source_dir, cat_name, cat_prefix in VIDEO_CATEGORIES:
    if not os.path.exists(source_dir):
        print(f"  Skipping {cat_name}"); continue

    all_vids = sorted([f for f in os.listdir(source_dir)
                       if f.lower().endswith((".mp4",".avi",".mov",".mkv"))])
    
    # -------------------------------------------------------------
    # STRATIFIED RANDOM SAMPLING
    # -------------------------------------------------------------
    # Balanced real-dataset evaluation: task uses snatching, normal, and crowded clips.
    target_count = min(CONFIG.get("sample_limits", {}).get(cat_name, len(all_vids)), len(all_vids))

    # Deterministic random sample for reproducible reports. Set EVALUATION_SEED differently only
    # when intentionally creating another run.
    sample = sorted(rng.sample(all_vids, target_count)) if target_count else []

    print(f"\n{'='*60}\n   {cat_name} - Sampled {len(sample)}/{len(all_vids)} videos (Seed: {EVALUATION_SEED})\n{'='*60}")

    for idx, vfile in enumerate(sample, 1):
        vpath   = os.path.join(source_dir, vfile)
        outpath = os.path.join(CONFIG["videos_out_dir"],
                               f"{cat_prefix}_{Path(vfile).stem}_final.avi")
        print(f"\n  [{idx}/{len(sample)}] {vfile}")
        
        try:
            log, snaps, vid_acc, mp4_out, harvested_features = run_detection(vpath, outpath, category_name=cat_name)
        except Exception:
            import traceback; traceback.print_exc(); continue

        for e in log:
            e["category"] = cat_name; e["video"] = vfile
            
        vid_acc["category"] = cat_name
        all_alert_logs.extend(log)
        all_snapshot_frames.extend(snaps)
        all_video_accuracy.append(vid_acc)
        MASTER_DATASET.extend(harvested_features)
        
        if mp4_out: generated_videos.append((cat_name, vfile, mp4_out))

print(f"\n{'='*60}")
print(f"  Videos processed : {len(generated_videos)}")
print(f"  Total alerts     : {len(all_alert_logs)}")
print(f"  Wiretapped rows  : {len(MASTER_DATASET)}")
print(f"{'='*60}\n")



----------------------------------------------------------------------
    PHASE 7 - Video Detection Loop  |  Elapsed: 0.2 min
----------------------------------------------------------------------

   Snatching - Sampled 20/74 videos (Seed: 42)

  [1/20] SN_02.mp4

 SN_02.mp4
  Zones: 1 | ['DEFAULT_RESTRICTED_ZONE']


  Detecting: 100%|██████████| 1573/1573 [12:07<00:00,  2.16fr/s]


  1573 frames | 27 violations | 2.2 fps
  CLAHE enhanced: 1573 frames (100.0%)
  Multi-scale:    9 frames (0.6%)
  DetRate:83.2%  Conf:0.668  Stability:0.693

  [2/20] SN_04.mp4

 SN_04.mp4
  Zones: 1 | ['DEFAULT_RESTRICTED_ZONE']


  Detecting: 100%|██████████| 299/299 [01:57<00:00,  2.55fr/s]


  299 frames | 8 violations | 2.6 fps
  CLAHE enhanced: 0 frames (0.0%)
  Multi-scale:    0 frames (0.0%)
  DetRate:57.2%  Conf:0.480  Stability:0.430

  [3/20] SN_05.mp4

 SN_05.mp4
  Zones: 1 | ['DEFAULT_RESTRICTED_ZONE']


...

  489 frames | 86 violations | 1.3 fps
  CLAHE enhanced: 390 frames (79.8%)
  Multi-scale:    0 frames (0.0%)
  DetRate:96.9%  Conf:0.726  Stability:0.809

  [9/10] Screen Recording 2026-04-01 210708.mp4

 Screen Recording 2026-04-01 210708.mp4
  Zones: 1 | ['DEFAULT_RESTRICTED_ZONE']


  Detecting: 100%|██████████| 672/672 [15:41<00:00,  1.40s/fr]


  672 frames | 628 violations | 0.7 fps
  CLAHE enhanced: 597 frames (88.8%)
  Multi-scale:    31 frames (4.6%)
  DetRate:98.1%  Conf:0.842  Stability:0.873

  [10/10] Screen Recording 2026-04-01 211716.mp4

 Screen Recording 2026-04-01 211716.mp4
  Zones: 1 | ['DEFAULT_RESTRICTED_ZONE']


  Detecting: 100%|██████████| 432/432 [06:39<00:00,  1.08fr/s]

  432 frames | 530 violations | 1.1 fps
  CLAHE enhanced: 432 frames (100.0%)
  Multi-scale:    88 frames (20.4%)
  DetRate:100.0%  Conf:0.713  Stability:0.870

  Videos processed : 40
  Total alerts     : 11267
  Wiretapped rows  : 11168



In [17]:
import pandas as pd
import os

# =====================================================================
# [NEW HARVESTER] Encode Category and Keep Features
# =====================================================================
print(f"Total interactions wiretapped: {len(harvested_features)}")

csv_data = []
for event in MASTER_DATASET:
    cat = str(event.get("category", "")).strip().lower()
    event["label"] = 1 if cat == "snatching" else 0
    csv_data.append(event)

if len(csv_data) > 0:
    df_features = pd.DataFrame(csv_data)
    
    # [NEW ENCODING LOGIC] Convert text category into numeric codes
    if "category" in df_features.columns:
        df_features["category_encoded"] = df_features["category"].astype("category").cat.codes
        df_features = df_features.drop(columns=["category"])

    csv_path = "/kaggle/working/outputs/labelled_interaction_features.csv"
    os.makedirs("/kaggle/working/outputs", exist_ok=True)
    df_features.to_csv(csv_path, index=False)

    print(f"- Success! Saved {len(df_features)} rows to {csv_path}")
    print(f"Label Split: {df_features['label'].value_counts().to_dict()}")
else:
    print("- No data collected.")


Total interactions wiretapped: 515
- Success! Saved 11168 rows to /kaggle/working/outputs/labelled_interaction_features.csv
Label Split: {0: 10721, 1: 447}


In [18]:
# Cell 13b ? Optional SVM training hook
# Description: This is the practical bridge to the paper's SVM stage. After you collect labeled interaction
# feature rows, set LABELLED_FEATURE_CSV to that file and run this cell to train /kaggle/working/snatch_svm.joblib.
# The detector will use it when CONFIG["svm_enabled"] = True.

LABELLED_FEATURE_CSV = "/kaggle/working/outputs/labelled_interaction_features.csv"
SVM_OUT = "/kaggle/working/snatch_svm.joblib"

if os.path.exists(LABELLED_FEATURE_CSV):
    import joblib
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import SVC
    from sklearn.model_selection import StratifiedKFold, cross_val_score

    df_feat = pd.read_csv(LABELLED_FEATURE_CSV)
    if "label" not in df_feat.columns:
        raise ValueError("label column missing. Use label=1 for snatch, label=0 for normal.")

    # ==========================================
    # [DATA BALANCING] Undersample the majority class
    # ==========================================
    df_majority = df_feat[df_feat["label"] == 0]
    df_minority = df_feat[df_feat["label"] == 1]
    
    # Calculate desired majority size (e.g., a 3-to-1 ratio)
    # 753 minority rows * 3 = 2,259 majority rows kept
    desired_majority_count = len(df_minority) * 3 
    
    if len(df_majority) > desired_majority_count:
        # Randomly sample the majority class down to the desired count
        df_majority_downsampled = df_majority.sample(n=desired_majority_count, random_state=42)
        # Re-combine the dataset and shuffle it
        df_feat = pd.concat([df_majority_downsampled, df_minority]).sample(frac=1, random_state=42).reset_index(drop=True)
        print(f"Dataset Balanced! New Label Split: {df_feat['label'].value_counts().to_dict()}")
    # ==========================================

    
    drop_cols = [c for c in ["label", "video", "frame", "pair"] if c in df_feat.columns]
    X = df_feat.drop(columns=drop_cols).fillna(0.0).astype(float).values
    y = df_feat["label"].astype(int).values
    svm = make_pipeline(StandardScaler(), 
                        SVC(kernel="rbf", C=3.0, gamma="scale", probability=True, class_weight="balanced")
                       )
    if len(set(y)) == 2 and len(y) >= 6:
        cv = StratifiedKFold(n_splits=min(3, np.bincount(y).min()), shuffle=True, random_state=42)
        print("SVM CV F1:", cross_val_score(svm, X, y, cv=cv, scoring="f1").round(3))
    svm.fit(X, y)
    joblib.dump(svm, SVM_OUT)
    print("Saved SVM model ->", SVM_OUT)
else:
    print("No labelled feature CSV found yet. The notebook will use calibrated hybrid rules.")
    print("Create:", LABELLED_FEATURE_CSV)


Dataset Balanced! New Label Split: {0: 1341, 1: 447}
SVM CV F1: [       0.99       0.987        0.99]
Saved SVM model -> /kaggle/working/snatch_svm.joblib


In [19]:
# Cell 14 - Save outputs + summary chart
alert_path = os.path.join(CONFIG["output_dir"], "alert_log.json")
with open(alert_path, "w") as f:
    json.dump(all_alert_logs, f, indent=2, default=str)
print(f" Alert log saved -> {alert_path}  ({len(all_alert_logs)} events)")

acc_path = os.path.join(CONFIG["output_dir"], "video_stability_summary.json")
with open(acc_path, "w") as f:
    json.dump(all_video_accuracy, f, indent=2, default=str)

acc_df  = pd.DataFrame([v for v in all_video_accuracy if v])
csv_path= os.path.join(CONFIG["output_dir"], "stability_report.csv")
acc_df.to_csv(csv_path, index=False)
print(f" Stability CSV -> {csv_path}")

for (fnum, frame, viols) in all_snapshot_frames:
    snap_path = os.path.join(CONFIG["output_dir"], f"violation_frame_{fnum}.jpg")
    cv2.imwrite(snap_path, frame)
print(f" Snapshots saved: {len(all_snapshot_frames)}")

cats   = ["Snatching", "NonSnatching", "Crowded"]
counts = [sum(v.get("total_violations",0) for v in all_video_accuracy if v.get("category")==c) for c in cats]
enh_pcts = [np.mean([v.get("enhanced_frames_pct",0) for v in all_video_accuracy if v.get("category")==c] or [0]) for c in cats]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(cats, counts, color=["#e74c3c","#2ecc71","#3498db"])
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 str(count), ha="center", fontsize=12, fontweight="bold")
axes[0].set_title("Violations Detected per Category", fontsize=13)
axes[0].set_ylabel("Total Violations")

bars2 = axes[1].bar(cats, enh_pcts, color=["#f39c12","#8e44ad","#1abc9c"])
for bar, pct in zip(bars2, enh_pcts):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f"{pct:.1f}%", ha="center", fontsize=12, fontweight="bold")
axes[1].set_title("CLAHE-Enhanced Frames % per Category", fontsize=13)
axes[1].set_ylabel("% of Frames Enhanced")
axes[1].set_ylim(0, 105)

plt.tight_layout()
chart_path = os.path.join(CONFIG["output_dir"], "violations_and_enhancement_chart.png")
plt.savefig(chart_path, dpi=150); plt.close()
print(f" Chart saved -> {chart_path}")

print("\n" + "="*60)
print("  FINAL RUN SUMMARY")
print("="*60)
print(f"  Videos processed : {len(all_video_accuracy)}")
print(f"  Total alerts     : {len(all_alert_logs)}")
for c, n in zip(cats, counts):
    print(f"  {c:<15}: {n} violations")
if acc_df is not None and len(acc_df):
    print(f"  Avg detection rate  : {acc_df['detection_rate'].mean():.1%}")
    print(f"  Avg confidence      : {acc_df['avg_conf'].mean():.3f}")
    print(f"  Avg stability index : {acc_df['stability_index'].mean():.3f}")
    if "enhanced_frames_pct" in acc_df.columns:
        print(f"  Avg CLAHE-enhanced  : {acc_df['enhanced_frames_pct'].mean():.1f}%")
    if "multiscale_frames_pct" in acc_df.columns:
        print(f"  Avg multi-scale     : {acc_df['multiscale_frames_pct'].mean():.1f}%")
print("="*60)

print("="*60)


 Alert log saved -> /kaggle/working/outputs/alert_log.json  (11267 events)
 Stability CSV -> /kaggle/working/outputs/stability_report.csv
 Snapshots saved: 176
 Chart saved -> /kaggle/working/outputs/violations_and_enhancement_chart.png

  FINAL RUN SUMMARY
  Videos processed : 40
  Total alerts     : 11267
  Snatching      : 464 violations
  NonSnatching   : 1598 violations
  Crowded        : 9205 violations
  Avg detection rate  : 84.6%
  Avg confidence      : 0.662
  Avg stability index : 0.729
  Avg CLAHE-enhanced  : 69.3%
  Avg multi-scale     : 17.5%


In [20]:
import pandas as pd

ground_truth_rows = []

# Snatching videos -> 1
for v in acc_df[acc_df["category"] == "Snatching"]["video"]:
    ground_truth_rows.append({
        "video": v,
        "ground_truth": 1
    })

# NonSnatching + Crowded -> 0
for v in acc_df[acc_df["category"] != "Snatching"]["video"]:
    ground_truth_rows.append({
        "video": v,
        "ground_truth": 0
    })

gt_df = pd.DataFrame(ground_truth_rows)

gt_df.to_csv(
    "/kaggle/working/outputs/ground_truth.csv",
    index=False
)

print("Ground Truth CSV Created")
print(gt_df.head())


Ground Truth CSV Created
       video  ground_truth
0  SN_02.mp4             1
1  SN_04.mp4             1
2  SN_05.mp4             1
3  SN_06.mp4             1
4  SN_12.mp4             1


In [21]:
#  for evaluation based on Ground truth 
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

gt_df = pd.read_csv(
    "/kaggle/working/outputs/ground_truth.csv"
)

eval_df = acc_df.merge(
    gt_df,
    on="video",
    how="inner"
)

eval_df["prediction"] = (
    eval_df["total_violations"] > 0
).astype(int)

gt = eval_df["ground_truth"]
pred = eval_df["prediction"]

print("="*60)
print("GROUND TRUTH EVALUATION")
print("="*60)

print(
    f"Accuracy  : {accuracy_score(gt,pred):.4f}"
)

print(
    f"Precision : {precision_score(gt,pred,zero_division=0):.4f}"
)

print(
    f"Recall    : {recall_score(gt,pred,zero_division=0):.4f}"
)

print(
    f"F1 Score  : {f1_score(gt,pred,zero_division=0):.4f}"
)

print("\nConfusion Matrix")
print(confusion_matrix(gt,pred))


GROUND TRUTH EVALUATION
Accuracy  : 0.5000
Precision : 0.5000
Recall    : 0.9500
F1 Score  : 0.6552

Confusion Matrix
[[ 1 19]
 [ 1 19]]


In [22]:
#  to save the evaluation csv
eval_df.to_csv(
    "/kaggle/working/outputs/evaluation_results.csv",
    index=False
)


In [23]:
import shutil, os

output_root = "/kaggle/working/outputs"
zip_path = "/kaggle/working/outputs_final"

# Zip the entire output folder
shutil.make_archive(zip_path, 'zip', output_root)
print(f"ZIP created: {zip_path}.zip")
print(f"Size: {os.path.getsize(zip_path + '.zip') / 1e6:.1f} MB")


ZIP created: /kaggle/working/outputs_final.zip
Size: 514.5 MB
